# 04 — Graph Dataset Build (MCMIPF → Graph Sequences)

## Purpose
This notebook constructs a reproducible spatio-temporal graph dataset for each site using GOES MCMIPF data.
The output is a graph store (one graph per timestamp) plus a manifest of supervised samples for a fixed
forecast horizon. The resulting artifacts are designed to be consumed later by a GraphSAGE + LSTM model.

**Key decisions (frozen for this dataset version)**
- Temporal grid: 10 minutes, UTC
- Forecast horizon: 6 hours (H = 36 steps)
- History length: 4 hours (L = 24 steps)
- Spatial patch: 32 × 32 centered at each site
- Graph connectivity: 8-neighborhood on the patch grid

## Import and settings

In [15]:
from __future__ import annotations

from pathlib import Path
import re
import numpy as np
import pandas as pd
import json

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path("..").resolve()

GROUND_DIR = PROJECT_ROOT / "data" / "ground_aligned"
UNI_GROUND = GROUND_DIR / "ground_10min_utc_uniandes.parquet"
ELP_GROUND = GROUND_DIR / "ground_10min_utc_elpaso.parquet"

MCMIPF_ROOT = PROJECT_ROOT / "data_processed" / "GOES_v2" / "MCMIPF"

OUT_ROOT = PROJECT_ROOT / "data" / "datasets" / "graph_v1"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

print("GROUND UNI:", UNI_GROUND.exists(), "->", UNI_GROUND)
print("GROUND ELP:", ELP_GROUND.exists(), "->", ELP_GROUND)
print("MCMIPF ROOT:", MCMIPF_ROOT.exists(), "->", MCMIPF_ROOT)

GROUND UNI: True -> /srv/projects/Proyecto_e_ladino/data/ground_aligned/ground_10min_utc_uniandes.parquet
GROUND ELP: True -> /srv/projects/Proyecto_e_ladino/data/ground_aligned/ground_10min_utc_elpaso.parquet
MCMIPF ROOT: True -> /srv/projects/Proyecto_e_ladino/data_processed/GOES_v2/MCMIPF


## Loading datasets

In [16]:
uni = pd.read_parquet(UNI_GROUND)
elp = pd.read_parquet(ELP_GROUND)

assert "ghi" in uni.columns and "ghi" in elp.columns, "Missing 'ghi' in ground datasets"
assert str(uni.index.tz) == "UTC", "UNI index is not UTC"
assert str(elp.index.tz) == "UTC", "ELP index is not UTC"

print("UNI:", uni.shape, "|", uni.index.min(), "→", uni.index.max())
print("ELP:", elp.shape, "|", elp.index.min(), "→", elp.index.max())

UNI: (82657, 6) | 2023-09-01 05:00:00+00:00 → 2025-03-28 05:00:00+00:00
ELP: (107172, 9) | 2022-02-21 18:00:00+00:00 → 2024-03-06 23:50:00+00:00


## Temporal splits

In [17]:
SPLITS = {
    "uniandes": {
        "train": ("2023-09-01", "2024-09-30"),
        "val":   ("2024-10-01", "2024-12-31"),
        "test":  ("2025-01-01", "2025-03-28"),
    },
    "elpaso": {
        "train": ("2022-03-01", "2023-06-30"),
        "val":   ("2023-07-01", "2023-10-31"),
        "test":  ("2023-11-01", "2024-03-07"),
    },
}

def slice_by_date(df: pd.DataFrame, start: str, end: str) -> pd.DataFrame:
    start_ts = pd.Timestamp(start, tz="UTC")
    end_ts = pd.Timestamp(end, tz="UTC")
    return df.loc[(df.index >= start_ts) & (df.index < end_ts)].copy()

def build_splits(df: pd.DataFrame, site_key: str) -> dict[str, pd.DataFrame]:
    spec = SPLITS[site_key]
    return {name: slice_by_date(df, s, e) for name, (s, e) in spec.items()}

uni_splits = build_splits(uni, "uniandes")
elp_splits = build_splits(elp, "elpaso")

for k, v in uni_splits.items():
    print("UNI", k, v.index.min(), "→", v.index.max(), "| rows:", len(v))
for k, v in elp_splits.items():
    print("ELP", k, v.index.min(), "→", v.index.max(), "| rows:", len(v))

UNI train 2023-09-01 05:00:00+00:00 → 2024-09-29 23:50:00+00:00 | rows: 56850
UNI val 2024-10-01 00:00:00+00:00 → 2024-12-30 23:50:00+00:00 | rows: 13104
UNI test 2025-01-01 00:00:00+00:00 → 2025-03-27 23:50:00+00:00 | rows: 12384
ELP train 2022-03-01 00:00:00+00:00 → 2023-06-29 23:50:00+00:00 | rows: 69984
ELP val 2023-07-01 00:00:00+00:00 → 2023-10-30 23:50:00+00:00 | rows: 17568
ELP test 2023-11-01 00:00:00+00:00 → 2024-03-06 23:50:00+00:00 | rows: 18288


## Temporal parameters

In [18]:
FREQ_MIN = 10
HOURS_AHEAD = 6
H = int((HOURS_AHEAD * 60) / FREQ_MIN)  # 36

HIST_HOURS = 4
L = int((HIST_HOURS * 60) / FREQ_MIN)   # 24

print("Horizon steps H:", H, f"({HOURS_AHEAD}h)")
print("History steps  L:", L, f"({HIST_HOURS}h)")

Horizon steps H: 36 (6h)
History steps  L: 24 (4h)


## Spatial parameters and site centers

- Patch size is fixed at 32x32 on the 256x256 grid.


In [19]:
GRID_SIZE = 256
PATCH = 32
HALF = PATCH // 2

META_PATH = PROJECT_ROOT / "data" / "metadata" / "site_center_pix_256.json"
assert META_PATH.exists(), f"Missing metadata JSON: {META_PATH}"

with open(META_PATH, "r", encoding="utf-8") as f:
    meta = json.load(f)

assert int(meta["grid_out"]) == GRID_SIZE, "Metadata grid_out does not match expected GRID_SIZE"

SITE_CENTER_PIX = {
    k: (int(v["row"]), int(v["col"]))
    for k, v in meta["sites"].items()
}

print("Loaded SITE_CENTER_PIX:")
for site, (r, c) in SITE_CENTER_PIX.items():
    print(f"  {site}: (row={r}, col={c})")

Loaded SITE_CENTER_PIX:
  uniandes: (row=160, col=49)
  elpaso: (row=91, col=53)


### Sanity checks

In [20]:
for site, (r, c) in SITE_CENTER_PIX.items():
    r1, r2 = r - HALF, r + HALF
    c1, c2 = c - HALF, c + HALF
    ok = (0 <= r1) and (0 <= c1) and (r2 <= GRID_SIZE) and (c2 <= GRID_SIZE)
    print(f"{site}: patch rows [{r1},{r2}) cols [{c1},{c2}) -> within grid: {ok}")
    if not ok:
        raise ValueError(f"Patch for {site} exceeds grid bounds. Adjust PATCH or site center.")

uniandes: patch rows [144,176) cols [33,65) -> within grid: True
elpaso: patch rows [75,107) cols [37,69) -> within grid: True


## MCMIPF

MCMIPF files are stored hourly with naming:

    YYYYMMDD_HH_MCMIPF.npz

Each file contains the array "mcmipf" with shape: (6, 16, 256, 256)

corresponding to 6 ten-minute frames within that hour, 16 channels, 256x256 grid.

In [21]:
def list_mcmipf_files(root: Path) -> list[Path]:
    return sorted(root.rglob("*_MCMIPF.npz"))

FILES = list_mcmipf_files(MCMIPF_ROOT)
print("Found MCMIPF files:", len(FILES))
print("Example:", FILES[0] if FILES else None)

RX = re.compile(r"(?P<ymd>\d{8})_(?P<h>\d{2})_MCMIPF\.npz$")

def parse_hour_key_from_name(p: Path) -> tuple[str, int] | None:
    m = RX.search(p.name)
    if not m:
        return None
    ymd = m.group("ymd")
    hh = int(m.group("h"))
    return (ymd, hh)

hourkey_to_path: dict[tuple[str, int], Path] = {}
bad = 0
for p in FILES:
    key = parse_hour_key_from_name(p)
    if key is None:
        bad += 1
        continue
    # keep first occurrence if duplicates exist
    if key not in hourkey_to_path:
        hourkey_to_path[key] = p

print("Parsed hour-container files:", len(hourkey_to_path))
print("Unparsed files:", bad)

Found MCMIPF files: 27436
Example: /srv/projects/Proyecto_e_ladino/data_processed/GOES_v2/MCMIPF/2022/02/20220201_00_MCMIPF.npz
Parsed hour-container files: 27436
Unparsed files: 0


### Coverage check
Timestamp → hourkey + slot (10-min grid)

In [22]:
def ts_to_hourkey_and_slot(t: pd.Timestamp) -> tuple[tuple[str, int], int]:
    ymd = t.strftime("%Y%m%d")
    hh = int(t.strftime("%H"))
    slot = int(t.strftime("%M")) // 10  # 0..5
    return (ymd, hh), slot

def coverage_check(ground_df: pd.DataFrame, label: str) -> None:
    ok = 0
    total = len(ground_df)
    for t in ground_df.index:
        key, slot = ts_to_hourkey_and_slot(t)
        if key in hourkey_to_path and 0 <= slot <= 5:
            ok += 1
    print(f"{label} satellite coverage: {ok}/{total} ({100*ok/total:.2f}%)")

coverage_check(uni, "UNI")
coverage_check(elp, "ELP")


UNI satellite coverage: 81553/82657 (98.66%)
ELP satellite coverage: 106050/107172 (98.95%)


### Graph topology

8-neighborhood on a PATCH×PATCH grid

In [23]:
def build_edge_index_8n(patch: int) -> np.ndarray:
    edges = []
    def node_id(rr: int, cc: int) -> int:
        return rr * patch + cc

    for rr in range(patch):
        for cc in range(patch):
            u = node_id(rr, cc)
            for dr in (-1, 0, 1):
                for dc in (-1, 0, 1):
                    if dr == 0 and dc == 0:
                        continue
                    r2 = rr + dr
                    c2 = cc + dc
                    if 0 <= r2 < patch and 0 <= c2 < patch:
                        v = node_id(r2, c2)
                        edges.append((u, v))

    return np.asarray(edges, dtype=np.int64).T  # (2, E)

EDGE_INDEX = build_edge_index_8n(PATCH)
print("EDGE_INDEX shape:", EDGE_INDEX.shape, "| edges:", EDGE_INDEX.shape[1])

EDGE_INDEX shape: (2, 7812) | edges: 7812


### Loading and extracting patch features

In [24]:
def load_mcmipf_tensor(path: Path) -> np.ndarray:
    d = np.load(path)
    if "mcmipf" not in d.files:
        raise KeyError(f"'mcmipf' key not found in {path.name}. Keys: {d.files}")
    return d["mcmipf"].astype(np.float32)  # (6, 16, 256, 256)

def extract_patch_from_frame_c_hw(frame_c_hw: np.ndarray, center_rc: tuple[int, int], patch: int) -> np.ndarray:
    r0, c0 = center_rc
    half = patch // 2
    r1, r2 = r0 - half, r0 + half
    c1, c2 = c0 - half, c0 + half

    if (r1 < 0) or (c1 < 0) or (r2 > GRID_SIZE) or (c2 > GRID_SIZE):
        raise ValueError(f"Patch exceeds bounds: rows [{r1},{r2}) cols [{c1},{c2})")

    patch_c_hw = frame_c_hw[:, r1:r2, c1:c2]  # (C, P, P)
    return patch_c_hw

def node_features_from_patch(patch_c_pp: np.ndarray) -> np.ndarray:
    C, P1, P2 = patch_c_pp.shape
    x = np.transpose(patch_c_pp, (1, 2, 0)).reshape(P1 * P2, C).astype(np.float32)
    return np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

### Graph store build

Graphs are stored per site under:

data/datasets/graph_v1/<site>/graphs/YYYYMMDD_HHMM.npz

In [25]:
def build_graph_store_for_site(site: str, ground_df: pd.DataFrame, out_dir: Path):
    out_graphs = out_dir / "graphs"
    out_graphs.mkdir(parents=True, exist_ok=True)

    r, c = SITE_CENTER_PIX[site]
    idx = ground_df.index

    written = 0
    skipped = 0

    for t in idx:
        out_path = out_graphs / f"{t.strftime('%Y%m%d_%H%M')}.npz"
        if out_path.exists():
            continue

        key, slot = ts_to_hourkey_and_slot(t)
        path = hourkey_to_path.get(key)
        if path is None:
            skipped += 1
            continue

        tensor = load_mcmipf_tensor(path)       # (6,16,256,256)
        frame = tensor[slot]                    # (16,256,256)
        patch_c_pp = extract_patch_from_frame_c_hw(frame, (r, c), PATCH)  # (16,P,P)
        x = node_features_from_patch(patch_c_pp) # (P*P,16)

        np.savez_compressed(out_path, x=x, edge_index=EDGE_INDEX)
        written += 1

    print(f"[{site}] graphs written: {written} | skipped (no sat file): {skipped}")
    print(f"[{site}] graphs stored in: {out_graphs}")

## Manifest construction

Each manifest row corresponds to a supervised sample:
 - t_label: current time
 - history_files: list of graph files for the history window [t-L+1, ..., t]
 - y: ground ghi at t_target = t + H steps

In [26]:
def manifest_for_site_split(
    site: str,
    split_df: pd.DataFrame,
    out_dir: Path,
    horizon_steps: int,
    history_steps: int,
) -> pd.DataFrame:
    graphs_dir = out_dir / "graphs"
    if not graphs_dir.exists():
        raise ValueError("Graph store does not exist. Build graphs first.")

    idx = split_df.index
    rows = []

    for t in idx:
        t_target = t + pd.Timedelta(minutes=FREQ_MIN * horizon_steps)
        t_hist_start = t - pd.Timedelta(minutes=FREQ_MIN * (history_steps - 1))

        if t_target not in idx:
            continue
        if pd.isna(split_df.loc[t_target, "ghi"]):
            continue

        hist = pd.date_range(t_hist_start, t, freq=f"{FREQ_MIN}min", tz="UTC")
        if len(hist) != history_steps:
            continue

        history_files = []
        ok = True
        for th in hist:
            fp = graphs_dir / f"{th.strftime('%Y%m%d_%H%M')}.npz"
            if not fp.exists():
                ok = False
                break
            history_files.append(str(fp))

        if not ok:
            continue

        rows.append({
            "site": site,
            "t_label": t,
            "t_target": t_target,
            "y": float(split_df.loc[t_target, "ghi"]),
            "history_files": history_files,
        })

    return pd.DataFrame(rows)



In [27]:
def build_all_manifests_for_site(site: str, splits: dict[str, pd.DataFrame], out_dir: Path):
    for split_name, df_split in splits.items():
        man = manifest_for_site_split(site, df_split, out_dir, H, L)
        out_path = out_dir / f"manifest_{split_name}.parquet"
        man.to_parquet(out_path, index=False) 
        print(f"[{site}] {split_name} manifest: {man.shape} -> {out_path}")

## Run: build graph stores and manifests

In [28]:
# uni_out_site = OUT_ROOT / "uniandes"
# elp_out_site = OUT_ROOT / "elpaso"
# uni_out_site.mkdir(parents=True, exist_ok=True)
# elp_out_site.mkdir(parents=True, exist_ok=True)

# build_graph_store_for_site("uniandes", uni, uni_out_site)
# build_graph_store_for_site("elpaso", elp, elp_out_site)

# build_all_manifests_for_site("uniandes", uni_splits, uni_out_site)
# build_all_manifests_for_site("elpaso", elp_splits, elp_out_site)